# 4. Installing security cameras: minimum vertex cover

Lecture: *Week 6 - Integer Linear Programming - Further Problems*, PDF pages 9–15. Week 05 demo 8 introduced
vertex cover; this is the lecture's example.

**The problem.** A school has eight corridor corners (vertices 1–8) joined by ten corridors (edges). A camera
at a corner watches every corridor that ends there. Place as few cameras as possible so that every corridor is
watched. $x_u = 1$ if there is a camera at corner $u$ (page 13):

minimise $\sum_{u \in V} x_u$ subject to $x_u + x_v \ge 1$ for every corridor $\{u, v\}$, $x_u \in \{0, 1\}$.

The corridors are 1–2, 1–8, 2–3, 2–5, 3–4, 4–5, 5–6, 5–8, 6–7 and 7–8.

**The matrix `A`.** `A[i][j] = 1` if corners $i+1$ and $j+1$ are joined by a corridor. Only the part above the
diagonal is filled in ($i < j$), so each corridor appears once and gives one constraint. In the code corners
1–8 are `x[0]` … `x[7]`.

**How to run it.** Run the first code cell first (it installs Pyomo and the HiGHS solver; the first time takes about a minute), then *Run → Run All Cells*, or run the cells one by one while you read.

**Expected result.** **4 cameras.** There are exactly two minimum covers, both on the slides: corners 2, 4, 6, 8
(page 14, "MATLAB Solution") and corners 1, 3, 5, 7 (page 15, "Another possibility"). HiGHS, in the browser and
on a desktop, picks 2, 4, 6, 8 (`x[1]`, `x[3]`, `x[5]`, `x[7]`); GLPK picks 1, 3, 5, 7 (`x[0]`, `x[2]`, `x[4]`,
`x[6]`). Both are optimal.

*Original script: `5 Source Files/Python/security_cameras_vertex_cover.py`. Runs in the browser; Pyomo models use HiGHS instead of GLPK with the same optimal value (4 cameras).*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

Load Pyomo's modelling tools.

In [ ]:
from pyomo.environ import *

Create the model, enter the graph as the matrix `A`, and add one binary variable per corner.

In [ ]:
# Create the model and the graph: A[i][j] = 1 if corners i+1 and j+1 share a corridor (i < j)
model = ConcreteModel()
A = [
    [0, 1, 0, 0, 0, 0, 0, 1],   # corner 1: corridors to 2 and 8
    [0, 0, 1, 0, 1, 0, 0, 0],   # corner 2: to 3 and 5
    [0, 0, 0, 1, 0, 0, 0, 0],   # corner 3: to 4
    [0, 0, 0, 0, 1, 0, 0, 0],   # corner 4: to 5
    [0, 0, 0, 0, 0, 1, 0, 1],   # corner 5: to 6 and 8
    [0, 0, 0, 0, 0, 0, 1, 0],   # corner 6: to 7
    [0, 0, 0, 0, 0, 0, 0, 1],   # corner 7: to 8
    [0, 0, 0, 0, 0, 0, 0, 0]    # corner 8: its corridors are already listed above
]

no_vars = len(A[0])	# one variable per corner: the number of columns of A

model.x = Var(range(no_vars), within=Binary)   # x[i] = 1 if a camera is at corner i+1

The objective: the number of cameras, minimised.

In [ ]:
# Create the optimization problem
model.obj = Objective(expr=sum(model.x[i] for i in range(no_vars)), sense=minimize)

One constraint per corridor: the two loops visit every entry of `A`, and each 1 adds "camera at one end or the other". Ten 1s in `A` give the ten constraints on page 13.

In [ ]:
# Define the inequality constraints: one per corridor (each 1 in A)
model.vertex = ConstraintList()
for i in range(no_vars):
    for j in range(no_vars):
        if A[i][j]:
            model.vertex.add(model.x[i] + model.x[j] >= 1)   # a camera at corner i+1 or j+1

HiGHS is the solver; unlike GLPK it runs in the browser.

In [ ]:
# Create a solver
solver = SolverFactory('appsi_highs')

`pprint()` prints the model: check the ten constraints against page 13 (`x[0]` is $x_1$, and so on).

In [ ]:
# Display the model
model.pprint()

Solve. In the browser HiGHS prints its log here even with `tee=False` (on a desktop it stays quiet); the *Solving report* at the end should say *Status Optimal*.

In [ ]:
# Solve the model
solver.solve(model, tee=False)  # Set tee=True to display solver output

`display()` prints where the cameras go (value 1) and the objective (the number of cameras).

In [ ]:
# Display the results
model.display()

## What do you notice?

Could 3 cameras ever be enough? Look for corridors that share no corner.

<details><summary>Answer</summary>

No. Corridors 1–2, 3–4, 5–6 and 7–8 share no corner, so no single camera watches two of them: each needs its
own camera, and at least 4 are needed. Four corridors with no common corner form a *matching* (page 17); the
size of any matching is a lower bound on the size of any vertex cover.
</details>